In [ ]:
# DS-A6 | Causal Claim and Sensitivity Analysis
# Cell 1 — Configuration and imports
from pathlib import Path
import hashlib, json, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

ROOT = Path.cwd()
# If launched from the notebooks/ folder, move to the project root.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for folder in ["data", "results", "figures"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

SEED = 6042
GREEN = "#16803c"
PINK = "#ec168c"
DARK_GREEN = "#116329"
GRAY = "#777777"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 12, "axes.labelsize": 10, "legend.fontsize": 8})
print("Project root:", ROOT)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| statsmodels:", sm.__version__)


In [ ]:
# Cell 2 — Load the synthetic worked-example data (create it deterministically if missing)
DATA_PATH = ROOT / "data" / "synthetic_tutoring_example.csv"
if not DATA_PATH.exists():
    rng = np.random.default_rng(SEED)
    n = 2400
    prior = rng.normal(65, 12, n)
    motivation = rng.normal(0, 1, n)
    family = rng.normal(0, 1, n)
    prob = 1 / (1 + np.exp(-(-0.25 + 0.035 * (prior - 65) + 0.65 * motivation + 0.25 * family)))
    treatment = rng.binomial(1, prob, n)
    study = np.maximum(0, 4 + 0.10 * (prior - 65) + 0.9 * motivation + 0.5 * treatment + rng.normal(0, 1.5, n))
    exam = 0.62 * prior + 3 * treatment + 2 * motivation + 0.8 * family + rng.normal(0, 7, n)
    prequiz = 0.75 * prior + 2.5 * motivation + 0.7 * family + rng.normal(0, 6, n)
    engagement = 0.35 * treatment + 0.12 * (exam - 40) + rng.normal(0, 1, n)
    selected = (engagement > np.quantile(engagement, 0.18)).astype(int)
    pd.DataFrame({
        "student_id": np.arange(1, n + 1), "prior_score": prior, "family_support": family,
        "tutoring": treatment, "study_hours_post": study, "exam_score": exam,
        "pre_treatment_quiz": prequiz, "engagement_selection": selected
    }).to_csv(DATA_PATH, index=False, float_format="%.6f")

df = pd.read_csv(DATA_PATH)
sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
display(df.head())
display(pd.DataFrame({"column": df.columns, "dtype": [str(x) for x in df.dtypes], "missing": df.isna().sum().values}))
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | SHA-256: {sha256}")


In [ ]:
# Cell 3 — Data checks and treatment/outcome overview
required = ["student_id", "prior_score", "family_support", "tutoring", "study_hours_post",
            "exam_score", "pre_treatment_quiz", "engagement_selection"]
missing_columns = sorted(set(required) - set(df.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert df["student_id"].is_unique, "Student IDs must be unique."
assert df["tutoring"].dropna().isin([0, 1]).all(), "Treatment must be coded 0/1."
assert df[required].notna().all().all(), "Missing values found in required analysis columns."

group_summary = df.groupby("tutoring").agg(
    n=("student_id", "size"),
    mean_exam_score=("exam_score", "mean"),
    sd_exam_score=("exam_score", "std"),
    mean_prior_score=("prior_score", "mean"),
    mean_family_support=("family_support", "mean")
).rename(index={0: "No tutoring", 1: "Tutoring"})
display(group_summary.round(3))
print("Checks passed. This is synthetic teaching data, not evidence from a real tutoring programme.")


In [ ]:
# Cell 4 — Estimation helper and core causal estimands
def fit_effect(outcome, covariates):
    cols = ["tutoring"] + list(covariates)
    model = sm.OLS(df[outcome], sm.add_constant(df[cols], has_constant="add")).fit(cov_type="HC3")
    low, high = model.conf_int().loc["tutoring"]
    return {
        "estimate": float(model.params["tutoring"]),
        "robust_se_HC3": float(model.bse["tutoring"]),
        "ci_low_95": float(low),
        "ci_high_95": float(high),
        "p_value": float(model.pvalues["tutoring"]),
        "n": int(model.nobs),
        "r_squared": float(model.rsquared),
        "covariates": ", ".join(covariates) if covariates else "None"
    }

naive = fit_effect("exam_score", [])
adjusted = fit_effect("exam_score", ["prior_score", "family_support"])
mediator_diagnostic = fit_effect("exam_score", ["prior_score", "family_support", "study_hours_post"])
negative_control = fit_effect("pre_treatment_quiz", ["prior_score", "family_support"])

analysis_rows = []
for label, result in [
    ("Unadjusted association", naive),
    ("Adjusted for measured pre-treatment covariates", adjusted),
    ("Post-treatment mediator included (diagnostic only)", mediator_diagnostic),
    ("Negative-control outcome: pre-treatment quiz", negative_control)
]:
    analysis_rows.append({"analysis": label, **result})
estimates = pd.DataFrame(analysis_rows)
estimates.to_csv(ROOT / "results" / "demo_estimates.csv", index=False)
display(estimates.round(4))
print("Interpretation boundary: these are illustrative regression estimates on synthetic data.")


In [ ]:
# Cell 5 — Estimate comparison plot with 95% robust confidence intervals
plot_rows = [("Unadjusted", naive), ("Pre-treatment\nadjusted", adjusted),
             ("Mediator included\n(diagnostic)", mediator_diagnostic)]
labels = [x[0] for x in plot_rows]
vals = np.array([x[1]["estimate"] for x in plot_rows])
low = np.array([x[1]["ci_low_95"] for x in plot_rows])
high = np.array([x[1]["ci_high_95"] for x in plot_rows])
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.errorbar(np.arange(len(vals)), vals, yerr=[vals-low, high-vals], fmt="o",
            capsize=5, color=GREEN, ecolor=PINK, lw=1.7, markersize=7)
ax.axhline(0, color=GRAY, linestyle="--", linewidth=1)
ax.set_xticks(np.arange(len(labels)), labels)
ax.set_ylabel("Tutoring coefficient (exam-score points)")
ax.set_title("Synthetic Example: Estimates Depend on Adjustment Set")
ax.grid(axis="y", alpha=.22)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "01_demo_estimate_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 6 — Omitted-confounding bias sensitivity analysis
# This is a transparent bias-shift scenario grid, not a universal sensitivity estimator.
bias = np.linspace(-4, 4, 81)
sensitivity = pd.DataFrame({
    "assumed_omitted_bias": bias,
    "bias_adjusted_effect": adjusted["estimate"] - bias,
    "ci_low_95_shifted": adjusted["ci_low_95"] - bias,
    "ci_high_95_shifted": adjusted["ci_high_95"] - bias
})
sensitivity.to_csv(ROOT / "results" / "demo_omitted_bias_sensitivity.csv", index=False)
display(sensitivity.iloc[::10].round(3))
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(sensitivity["assumed_omitted_bias"], sensitivity["bias_adjusted_effect"],
        color=GREEN, lw=2, label="Bias-adjusted point estimate")
ax.fill_between(sensitivity["assumed_omitted_bias"], sensitivity["ci_low_95_shifted"],
                sensitivity["ci_high_95_shifted"], color=PINK, alpha=.18, label="Shifted 95% CI")
ax.axhline(0, color=GRAY, linestyle="--", linewidth=1)
ax.axvline(0, color=GRAY, linestyle=":", linewidth=1)
ax.set(title="Illustrative Omitted-Confounding Sensitivity",
       xlabel="Assumed omitted-variable bias (score points)",
       ylabel="Adjusted effect after assumed bias")
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "02_demo_sensitivity_curve.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 7 — Outcome distributions and overlap checks
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.hist(df.loc[df.tutoring == 0, "exam_score"], bins=35, alpha=.55, color=GREEN, label="No tutoring")
ax.hist(df.loc[df.tutoring == 1, "exam_score"], bins=35, alpha=.45, color=PINK, label="Tutoring")
ax.set(title="Outcome Distributions by Observed Treatment (Synthetic)",
       xlabel="Exam score", ylabel="Students")
ax.legend()
ax.grid(axis="y", alpha=.15)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "03_demo_outcome_distributions.png", dpi=220, bbox_inches="tight")
plt.show()

# Basic overlap check for measured pre-treatment variables by treatment group.
overlap_rows = []
for col in ["prior_score", "family_support"]:
    a = df.loc[df.tutoring == 0, col]
    b = df.loc[df.tutoring == 1, col]
    overlap_rows.append({
        "variable": col, "control_min": a.min(), "control_max": a.max(),
        "treated_min": b.min(), "treated_max": b.max(),
        "range_overlap": max(0.0, min(a.max(), b.max()) - max(a.min(), b.min()))
    })
overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table.round(3))


In [ ]:
# Cell 8 — DAG diagram: assumed causal structure for the synthetic example
# Arrows represent the illustrative data-generating assumptions, not proof from the data.
fig, ax = plt.subplots(figsize=(9, 5.3))
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.axis("off")
nodes = {
    "Prior score": (1.6, 5.5), "Family support": (1.6, 2.0),
    "Motivation\n(unobserved)": (1.6, 3.8), "Tutoring": (4.4, 5.5),
    "Study hours\n(post-treatment)": (6.7, 4.1), "Exam score": (8.6, 5.5),
    "Pre-treatment\nquiz (negative control)": (4.4, 1.5)
}
for label, (x, y) in nodes.items():
    fc = "#e8f5e9" if label in ("Tutoring", "Exam score") else "#fff0f7" if "unobserved" in label else "#f2f2f2"
    ax.text(x, y, label, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.45", fc=fc, ec=GREEN if fc=="#e8f5e9" else GRAY, lw=1.2))

def arrow(a, b, rad=0.0):
    x1, y1 = nodes[a]; x2, y2 = nodes[b]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=DARK_GREEN, lw=1.4,
                                connectionstyle=f"arc3,rad={rad}", shrinkA=35, shrinkB=35))
for a, b, rad in [
    ("Prior score", "Tutoring", 0), ("Prior score", "Exam score", -.12),
    ("Family support", "Tutoring", -.12), ("Family support", "Exam score", .10),
    ("Motivation\n(unobserved)", "Tutoring", -.08), ("Motivation\n(unobserved)", "Exam score", .12),
    ("Tutoring", "Study hours\n(post-treatment)", 0), ("Study hours\n(post-treatment)", "Exam score", 0),
    ("Prior score", "Pre-treatment\nquiz (negative control)", .10),
    ("Family support", "Pre-treatment\nquiz (negative control)", -.10),
    ("Motivation\n(unobserved)", "Pre-treatment\nquiz (negative control)", 0)
]:
    arrow(a, b, rad)
ax.set_title("Illustrative DAG for the Synthetic Tutoring Example", pad=12)
fig.tight_layout()
fig.savefig(ROOT / "figures" / "04_demo_dag.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 9 — Assumptions, diagnostics, and reproducibility record
assumptions = pd.DataFrame([
    ["Consistency", "Tutoring is a well-defined intervention with comparable versions", "Not established by synthetic observational rows alone"],
    ["Conditional exchangeability", "Measured pre-treatment covariates block confounding", "Motivation is unobserved in this demonstration; residual confounding remains plausible"],
    ["Positivity", "Both treatment states occur across covariate strata", "Only basic observed-range overlap was checked; inspect propensity-score overlap in real work"],
    ["No interference", "One student's treatment does not affect another student's outcome", "Not tested here"],
    ["Correct temporal ordering", "Adjustment variables precede treatment unless explicitly diagnostic", "Study hours are post-treatment and should not be adjusted for when estimating the total effect"],
    ["Negative control", "A pre-treatment outcome should not be caused by current tutoring", "Use as a refutation diagnostic, not a definitive test"]
], columns=["assumption", "meaning", "status / limitation"])
display(assumptions)
assumptions.to_csv(ROOT / "results" / "demo_assumptions_table.csv", index=False)

environment = {
    "purpose": "synthetic causal-method demonstration only",
    "seed": SEED, "python": sys.version, "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "statsmodels": sm.__version__,
    "dataset_file": str(DATA_PATH.relative_to(ROOT)), "dataset_sha256": sha256,
    "n_rows": int(len(df)), "known_treatment_effect_in_data_generator": 3.0,
    "colors": {"GREEN": GREEN, "PINK": PINK, "DARK_GREEN": DARK_GREEN}
}
(ROOT / "results" / "demo_environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))


In [ ]:
# Cell 10 — Final output inventory and short interpretation scaffold
outputs = sorted(str(p.relative_to(ROOT)) for p in (ROOT / "results").glob("*") if p.is_file())
figures = sorted(str(p.relative_to(ROOT)) for p in (ROOT / "figures").glob("*.png"))
print("RESULT FILES")
for item in outputs: print(" -", item)
print("\nFIGURES")
for item in figures: print(" -", item)

print("\nINTERPRETATION SCAFFOLD")
print(f"Unadjusted estimate: {naive['estimate']:.3f} (95% CI {naive['ci_low_95']:.3f}, {naive['ci_high_95']:.3f})")
print(f"Pre-treatment adjusted estimate: {adjusted['estimate']:.3f} (95% CI {adjusted['ci_low_95']:.3f}, {adjusted['ci_high_95']:.3f})")
print("Do not infer causation from the association alone. Discuss confounding, identification assumptions, overlap, negative controls, sensitivity, fairness, cost, appeal, and stop conditions.")
print("This synthetic example is for learning only; replace it with a verified dataset and a justified estimand for your assessed project.")
